# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, leveraging Croissant schemas to handle metadata, data extraction, and advanced analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show summary
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}\nAuthors: {metadata.author}")

## 2. Data Overview
Review available record sets, their fields, and `@id` values.
All entity references below use the `@id` as required.

In [ ]:
# List available record sets and fields with their @id values
from pprint import pprint

record_sets = []
if hasattr(metadata, 'record_set') and metadata.record_set:
    # Standard Croissant property
    record_sets = metadata.record_set
elif hasattr(metadata, 'recordSet') and metadata.recordSet:
    # Some Croissant schemas use PascalCase
    record_sets = metadata.recordSet

if not record_sets:
    # Try the fallback method for inspecting all keys
    record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Record Sets found:")
    for rs in record_sets:
        print(f"  - Name: {getattr(rs, 'name', None)} | @id: {getattr(rs, '@id', None)}")
        if hasattr(rs, 'field') and rs.field:
            print("    Fields:")
            for field in rs.field:
                print(f"      - {getattr(field, 'name', None)} | @id: {getattr(field, '@id', None)} | dataType: {getattr(field, 'data_type', getattr(field, 'dataType', None))}")

Let's inspect a sample record from *each* record set using its `@id`. 

**Note**: If the dataset has a single main record set, we'll use that one as an example.

In [ ]:
# List all record set ids for easy reference
record_set_ids = []
if record_sets:
    record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
    print('Record Set @id values:')
    pprint(record_set_ids)

# Show a single sample record from each available record set
for record_set_id in record_set_ids:
    print(f"\nSample record from Record Set '@id': {record_set_id}")
    try:
        for i, record in enumerate(dataset.records(record_set=record_set_id)):
            pprint(record)
            if i >= 0:  # print only first example
                break
    except Exception as e:
        print(f"  [Error extracting record set {record_set_id}: {e}]")

## 3. Data Extraction
Load data for each record set into pandas DataFrames for downstream analysis. 
All record set and field references use explicit `@id` values.


In [ ]:
# Extract available record sets into DataFrames, using @id keys
dataframes = {}
# Use record_set_ids determined before; fall back to manual if needed
if not record_set_ids and hasattr(metadata, 'recordSet'):
    record_set_ids = [getattr(rs, '@id', None) for rs in metadata.recordSet]

for rs_id in record_set_ids:
    print(f"Loading data from record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Shape: {df.shape}")
            print(f"  Columns: {df.columns.tolist()}")
        else:
            print("  [No records found]")
    except Exception as e:
        print(f"  [Error extracting {rs_id}: {e}]")

# Select a primary record set to preview (first one, if available)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nPreview (head) of record set: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering by value, normalizing numeric columns, or grouping by attributes by referencing fields using their `@id`.

In [ ]:
# Example EDA: filter, normalize, and group by fields using their @id.

# Use the main dataframe extracted previously
df = dataframes.get(main_record_set_id)

# Attempt to pick a likely numeric field (by @id or name)
# If you know the @id of your numeric fields, set it here! Otherwise, use an example.
numeric_field_id = None
group_field_id = None

if df is not None:
    # Try to automatically detect a likely numeric field (by dtype)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found, please adjust numeric_field_id.")
    else:
        print(f"Using numeric field: {numeric_field_id}")

    # Try to find a field for grouping (categorical, but not the numeric we picked)
    for col in df.columns:
        if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by field: {group_field_id}")

    # Filtering: show only records with numeric_field > threshold
    threshold = 10
    try:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
    except Exception as e:
        print(f"Error filtering: {e}")

    # Normalization
    try:
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    except Exception as e:
        print(f"Normalization failed: {e}")

    # Grouping by group_field_id
    if group_field_id:
        try:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped (mean) by {group_field_id}:")
            display(grouped_df.head())
        except Exception as e:
            print(f"Error in grouping: {e}")

## 5. Visualization
Visualize distributions of key fields or relationships using Matplotlib or Seaborn.
All plots use fields referenced by their `@id` per the Croissant model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure DataFrame and field ids are set
if df is not None and numeric_field_id is not None:
    # Distribution of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Scatter plot versus group or another numeric (if available)
    other_numeric = [c for c in df.columns if c != numeric_field_id and pd.api.types.is_numeric_dtype(df[c])]
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=60)
        plt.show()
    elif other_numeric:
        plt.figure(figsize=(6,4))
        sns.scatterplot(x=df[other_numeric[0]], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} vs {other_numeric[0]}")
        plt.show()


## 6. Conclusion
In this notebook, we loaded and explored a rich clinical dataset describing second primary colorectal cancers in survivors, using the Croissant schema and `mlcroissant`. We demonstrated:
- How to examine metadata structure entirely by Croissant `@id`.
- Data extraction and normalization steps for numeric fields using only their `@id`.
- How to obtain a quick statistical and visual summary.

The FAIR^2 dataset is suitable for exploring clinicopathological and molecular features of colorectal cancer in survivors, and provides a reproducible, schema-driven structure for biomedical data science tasks.